# Introduction

This comprehensive notebook demonstrates the **multimodal capabilities** of **Google's Gemini API** using the latest models and features.

## What You'll Learn

### API Fundamentals
- Single vs batch processing
- Streaming responses
- Configuration options
- Error handling

### Text Capabilities (10 examples)
- Sentiment analysis, classification, NER
- Summarization, Q&A, translation
- Text completion and rewriting
- Entity and keyword extraction

### Vision Capabilities (8 examples)
- Object counting and visual Q&A
- Chart and document analysis
- Image captioning and comparison
- Visual reasoning and diagrams

### Video Capabilities (3 examples)
- Video understanding and analysis
- Frame-by-frame analysis
- Action recognition

### Audio Capabilities (2 examples)
- Speech transcription
- Audio content analysis

### PDF Capabilities (2 examples)
- Document analysis
- Multi-page extraction

### Advanced Features (8 examples)
- Structured JSON output
- Function calling
- Code execution
- Search grounding
- Long context (1M tokens)
- Code generation
- Mathematical and scientific reasoning
- Creative writing

## Model Used

**Model:** `gemini-2.0-flash-thinking-exp-1219`
- **Input modalities:** Text, Image, Video, Audio, PDF
- **Output:** Text
- **Context:** 1M+ tokens input, 65K+ tokens output
- **Special features:** Code execution, search grounding, thinking mode

# Setup and Configuration

In [1]:
# Install required packages
# !pip install google-genai pillow requests matplotlib pandas numpy

In [2]:
import os
import json
import time
from pathlib import Path
from typing import List, Dict, Any
from google import genai
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
import base64
import matplotlib.pyplot as plt
import numpy as np

# Check for API key
if 'GEMINI_API_KEY' not in os.environ:
    raise ValueError(
        "GEMINI_API_KEY not found in environment.\n"
        "Set it with: export GEMINI_API_KEY='your-key'\n"
        "Get your key at: https://aistudio.google.com/apikey"
    )

# Initialize client (new SDK)
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

print(" Gemini client initialized successfully")
print("Using google-genai SDK (new version)")

# Note: We'll use gemini-2.0-flash-thinking-exp-1219 as the default model
MODEL = "gemini-2.0-flash-thinking-exp-1219"
print(f"Default model: {MODEL}")

/Users/nipun/base/lib/python3.12/site-packages/pydantic/_internal/_fields.py:186: UserWarning: Field name "name" shadows an attribute in parent "Operation"; 
  warnings.warn(
/Users/nipun/base/lib/python3.12/site-packages/pydantic/_internal/_fields.py:186: UserWarning: Field name "metadata" shadows an attribute in parent "Operation"; 
  warnings.warn(
/Users/nipun/base/lib/python3.12/site-packages/pydantic/_internal/_fields.py:186: UserWarning: Field name "done" shadows an attribute in parent "Operation"; 
  warnings.warn(
/Users/nipun/base/lib/python3.12/site-packages/pydantic/_internal/_fields.py:186: UserWarning: Field name "error" shadows an attribute in parent "Operation"; 
  warnings.warn(


 Gemini client initialized successfully
Using google-genai SDK (new version)
Default model: gemini-2.0-flash-thinking-exp-1219


## Helper Functions

In [3]:
def print_section(title: str):    """Print formatted section header."""    print("\n" + "="*80)    print(title)    print("="*80)def print_result(label: str, content: str, indent: int = 0):    """Print formatted result."""    prefix = "  " * indent    print(f"{prefix}{label}: {content}")def load_image_from_url(url: str) -> Image.Image:    """Load an image from a URL."""    response = requests.get(url)    return Image.open(BytesIO(response.content))def create_sample_image(text: str, size=(800, 600)) -> Image.Image:    """Create a simple image with text for testing."""    img = Image.new('RGB', size, color='white')    draw = ImageDraw.Draw(img)    draw.text((50, size[1]//2), text, fill='black')    return imgprint("Helper functions loaded")

NameError: name 'title' is not defined

# Part 1: API Usage Patterns

## Single Request vs Batch Processing

In [ ]:
print("Single Request")
print("="*80)

# Single request - simplest way
response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents="What is the capital of France?")
print_result("Question", "What is the capital of France?")
print_result("Answer", response.text)

In [ ]:
print("\n" + "="*80)
print("Batch Processing")
print("="*80)

# Process multiple prompts efficiently
prompts = [
    "Translate 'Hello' to Spanish",
    "Translate 'Goodbye' to French",
    "Translate 'Thank you' to German",
    "Translate 'Welcome' to Italian"
]

# Method 1: Sequential (simple but slower)
print("Sequential processing:")
start = time.time()
results_seq = []
for prompt in prompts:
    response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
    results_seq.append(response.text.strip())
time_seq = time.time() - start

for i, (prompt, result) in enumerate(zip(prompts, results_seq), 1):
    print(f"  {i}. {prompt} → {result}")
print(f"Time: {time_seq:.2f}s")

## Configuration and Safety Settings

In [ ]:
print("\n" + "="*80)print("Streaming Responses")print("="*80)# Stream responses for long-running tasksprompt = """Write a detailed explanation of how neural networks work,covering architecture, training process, and applications."""print("Streaming response (token by token):\n")response = client.models.generate_content_stream(    model="gemini-2.0-flash-thinking-exp-1219",    contents=prompt)for chunk in response:    if chunk.text:        print(chunk.text, end='', flush=True)print("\n\nStreaming complete!")

# Part 2: Text-Only Tasks (10 tasks)

In [ ]:
print("\n" + "="*80)
print("Zero-Shot Sentiment Analysis")
print("="*80)

texts = [
    "This product is absolutely amazing! Best purchase I've made all year.",
    "Terrible experience. Waste of money and time.",
    "It's okay. Nothing special but does the job.",
    "I'm disappointed with the quality. Expected much better.",
    "Exceeded all my expectations! Highly recommend!"
]

prompt_template = """Classify the sentiment: Positive, Negative, or Neutral.
Reply with ONLY the sentiment label.

Text: {text}
Sentiment:"""

for i, text in enumerate(texts, 1):
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt_template.format(text=text)
    )
    sentiment = response.text.strip()
print(f"{i}. '{text[:50]}...'")
print(f"   → {sentiment}\n")

## Few-Shot Text Classification

In [ ]:
print("\n" + "="*80)
print("Few-Shot Text Classification")
print("="*80)

# Intent classification with examples
prompt = """Classify customer service queries into categories.

Examples:
"How do I reset my password?" → Technical Support
"I was charged twice" → Billing
"What are your hours?" → General Inquiry
"This is broken" → Complaint
"I want to cancel" → Account Management

Query: "{query}"
Category:"""

test_queries = [
    "My app keeps crashing when I upload photos",
    "Why was I charged for premium when I'm on free plan?",
    "Do you ship to Canada?",
    "The product arrived damaged",
    "How do I delete my account?"
]

for query in test_queries:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt.format(query=query)
    )
print(f"Query: {query}")
print(f"Category: {response.text.strip()}\n")

## Named Entity Recognition

In [ ]:
print("\n" + "="*80)
print("Named Entity Recognition (NER)")
print("="*80)

text = """Apple Inc. CEO Tim Cook announced a $500 million investment in renewable 
energy projects across California next month. The announcement was made at the 
company's headquarters in Cupertino on December 15, 2024."""

prompt = f"""Extract all named entities and categorize them:
PERSON, ORGANIZATION, LOCATION, MONEY, DATE

Text: {text}

Format as JSON with entity type as key."""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
print(f"Text: {text}\n")
print("Entities:")
print(response.text)

## Text Summarization

In [ ]:
print("\n" + "="*80)
print("Text Summarization")
print("="*80)

article = """Artificial intelligence continues to transform industries worldwide. Recent
advances in large language models have enabled more natural conversations between humans
and machines. These models can understand context, generate coherent text, and even
perform complex reasoning tasks. However, challenges remain in ensuring factual accuracy,
reducing computational costs, and addressing ethical concerns around bias and privacy.
Researchers are actively working on making AI more efficient, transparent, and aligned
with human values. The field is evolving rapidly, with new breakthroughs announced weekly.
From healthcare to education, AI is reshaping how we work and live."""

prompts = [
    "Summarize in 1 sentence:",
    "Summarize in 3 bullet points:",
    "Create a tweet-length summary (280 chars):"
]

print(f"Original ({len(article)} chars):\n{article}\n")

for prompt_type in prompts:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=f"{prompt_type}\n\n{article}"
    )
print(f"{prompt_type}")
print(f"  {response.text.strip()}\n")

## Question Answering with Context

In [ ]:
print("\n" + "="*80)
print("Question Answering")
print("="*80)

context = """The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars
in Paris, France. It was constructed from 1887 to 1889 as the centerpiece of the 1889
World's Fair. The tower is 330 meters (1,083 feet) tall, about the same height as an
81-story building. It was the tallest man-made structure in the world until the Chrysler
Building was completed in New York in 1930."""

questions = [
    "When was the Eiffel Tower built?",
    "How tall is the Eiffel Tower?",
    "Where is it located?",
    "What material is it made of?",
    "When did it stop being the tallest structure?"
]

print(f"Context: {context}\n")

for q in questions:
    prompt = f"Context: {context}\n\nQuestion: {q}\nAnswer (concise):"
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
print(f"Q: {q}")
print(f"A: {response.text.strip()}\n")

## Translation

In [ ]:
print("\n" + "="*80)
print("Multi-Language Translation")
print("="*80)

text = "Artificial intelligence is changing the world."
languages = ["Spanish", "French", "German", "Japanese", "Hindi", "Arabic"]

print(f"Original (English): {text}\n")

for lang in languages:
    prompt = f"Translate to {lang}: {text}"
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
print(f"{lang}: {response.text.strip()}")

## Text Completion

In [ ]:
print("\n" + "="*80)
print("Text Completion")
print("="*80)

prompts = [
    "The secret to happiness is",
    "In the year 2050, technology will",
    "The most important skill for the future is"
]

for prompt in prompts:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=f"Complete this sentence in 1-2 sentences: {prompt}"
    )
print(f"Prompt: '{prompt}'")
print(f"Completion: {response.text.strip()}\n")

## Entity Extraction

In [ ]:
print("\n" + "="*80)
print("Structured Entity Extraction")
print("="*80)

resume = """JOHN DOE
john.doe@email.com | (555) 123-4567 | linkedin.com/in/johndoe

EXPERIENCE
Senior Software Engineer, TechCorp (2020-Present)
- Led team of 5 engineers in developing cloud infrastructure
- Expertise: Python, AWS, Docker, Kubernetes

EDUCATION
M.S. Computer Science, Stanford University (2018)
B.S. Computer Science, MIT (2016)"""

prompt = f"""Extract key information as JSON:
{{
  "name": "",
  "email": "",
  "phone": "",
  "current_role": "",
  "company": "",
  "skills": [],
  "education": []
}}

Resume:
{resume}

Return only valid JSON:"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
print("Extracted data:")
print(response.text)

## Keyword Extraction

In [ ]:
print("\n" + "="*80)
print("Keyword Extraction")
print("="*80)

text = """Machine learning and deep learning are subsets of artificial intelligence 
that focus on training algorithms to recognize patterns in data. Neural networks, 
inspired by biological neurons, form the basis of deep learning systems. These 
technologies power applications like computer vision, natural language processing, 
and autonomous vehicles."""

prompt = f"""Extract the 5 most important keywords from this text.
Return as a comma-separated list.

Text: {text}

Keywords:"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
print(f"Text: {text}\n")
print(f"Keywords: {response.text.strip()}")

## Text Rewriting

In [ ]:
print("\n" + "="*80)
print("Text Rewriting for Different Audiences")
print("="*80)

original = """The algorithm leverages advanced neural architectures to optimize
multi-dimensional parameter spaces through stochastic gradient descent."""

audiences = [
    "Explain to a 10-year-old",
    "Rewrite for a business executive",
    "Simplify for general audience",
    "Make it poetic"
]

print(f"Original: {original}\n")

for audience in audiences:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=f"{audience}:\n\n{original}"
    )
print(f"{audience}:")
print(f"  {response.text.strip()}\n")

# Part 3: Vision Tasks (8 tasks)

## Object Counting

In [ ]:
print("\n" + "="*80)
print("Object Counting in Images")
print("="*80)

# Create a test image with multiple objects
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Draw different shapes
circles = [(2, 2), (5, 5), (8, 3), (3, 7), (7, 8)]
squares_x = [1, 6, 9]
squares_y = [5, 2, 7]

for x, y in circles:
    circle = plt.Circle((x, y), 0.3, color='red', alpha=0.7)
    ax.add_patch(circle)

from matplotlib.patches import Rectangle
for x, y in zip(squares_x, squares_y):
    square = Rectangle((x-0.3, y-0.3), 0.6, 0.6, color='blue', alpha=0.7)
    ax.add_patch(square)

plt.title('Count the Objects', fontsize=16)
plt.savefig('/tmp/objects.png', dpi=150, bbox_inches='tight')


In [ ]:
plt.close()

image = Image.open('/tmp/objects.png')

prompt = """Count the objects in this image:
1. How many red circles?
2. How many blue squares?
3. Total number of objects?"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, image])
print(response.text)

## Visual Question Answering (VQA)

In [ ]:
print("\n" + "="*80)
print("Visual Question Answering")
print("="*80)

# Use sample images from URLs
try:
    image_url = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800"
    image = load_image_from_url(image_url)

    # Show image
    plt.imshow(image)
    
    questions = [
        "What is the dominant color in this image?",
        "Describe the scenery",
        "What time of day does it appear to be?",
        "What mood does this image convey?"
    ]

    for q in questions:
        response = client.models.generate_content(
            model="gemini-2.0-flash-thinking-exp-1219",
            contents=[q, image]
        )
print(f"Q: {q}")
print(f"A: {response.text.strip()}\n")

except Exception as e:
    print(f"Note: Image loading requires internet. Error: {str(e)[:100]}")

## Chart and Graph Analysis

In [ ]:
print("\n" + "="*80)
print("Chart Analysis")
print("="*80)

# Create a complex chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
sales = [45000, 52000, 48000, 61000, 58000, 72000]
ax1.bar(months, sales, color='steelblue')
ax1.set_title('Monthly Sales 2024', fontsize=14, fontweight='bold')
ax1.set_ylabel('Sales ($)')
ax1.grid(axis='y', alpha=0.3)

# Line chart
days = list(range(1, 31))
visitors = [100 + 50*np.sin(x/5) + np.random.randint(-10, 10) for x in days]
ax2.plot(days, visitors, marker='o', linewidth=2, markersize=4)
ax2.set_title('Daily Website Visitors', fontsize=14, fontweight='bold')
ax2.set_xlabel('Day of Month')
ax2.set_ylabel('Visitors')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/charts.png', dpi=150)


In [ ]:
plt.close()

chart_image = Image.open('/tmp/charts.png')

prompt = """Analyze these charts:
1. What trends do you see in the sales data?
2. Which month had the highest sales?
3. What pattern is visible in the website visitors chart?
4. Any notable insights?"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, chart_image])
print(response.text)

## Document OCR and Understanding

In [ ]:
print("\n" + "="*80)
print("Document OCR + Understanding")
print("="*80)

# Create a sample receipt
img = Image.new('RGB', (600, 800), color='white')
draw = ImageDraw.Draw(img)

receipt_lines = [
    "ACME STORE",
    "123 Main Street",
    "Phone: (555) 123-4567",
    "",
    "Date: 2024-12-01",
    "Receipt #: 45678",
    "-" * 40,
    "Coffee Beans (2kg)      $24.99",
    "Milk (1L)                $3.49",
    "Bread                    $2.99",
    "Fresh Vegetables        $12.50",
    "-" * 40,
    "Subtotal:              $43.97",
    "Tax (8%):               $3.52",
    "TOTAL:                 $47.49",
    "",
    "Payment: VISA ****1234",
    "Thank you for shopping!"
]

y = 50
for line in receipt_lines:
    draw.text((50, y), line, fill='black')
    y += 35

img.save('/tmp/receipt.png')
receipt_img = Image.open('/tmp/receipt.png')
plt.imshow(receipt_img)
plt.axis('off')



In [ ]:
prompt = """Extract information from this receipt:
1. Store name and address
2. Date and receipt number
3. List of items purchased with prices
4. Total amount
5. Payment method

Format as structured JSON."""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, receipt_img])
print(response.text)

## Image Captioning

In [ ]:
print("\n" + "="*80)
print("Image Captioning (Multiple Styles)")
print("="*80)

try:
    image_url = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800"
    image = load_image_from_url(image_url)
    # Show image
    plt.imshow(image)
    plt.axis('off')
    
    
    caption_styles = [
        "Write a short caption (1 sentence)",
        "Write a detailed caption (2-3 sentences)",
        "Write an Instagram caption with hashtags",
        "Write a poetic caption"
    ]

    for style in caption_styles:
        response = client.models.generate_content(
            model="gemini-2.0-flash-thinking-exp-1219",
            contents=[style, image]
        )
print(f"{style}:")
print(f"  {response.text.strip()}\n")

except Exception as e:
    print(f"Using local image. Error: {str(e)[:100]}")

## Multi-Image Comparison

In [ ]:
print("\n" + "="*80)
print("Multi-Image Comparison")
print("="*80)

# Create two different chart images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Image 1: Pie chart
sizes = [30, 25, 20, 15, 10]
labels = ['A', 'B', 'C', 'D', 'E']
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%')
axes[0].set_title('Product Distribution - Q1')

# Image 2: Pie chart with different values
sizes2 = [35, 20, 25, 10, 10]
axes[1].pie(sizes2, labels=labels, autopct='%1.1f%%')
axes[1].set_title('Product Distribution - Q2')

plt.tight_layout()
plt.savefig('/tmp/comparison.png', dpi=150)


In [ ]:
plt.close()

comp_image = Image.open('/tmp/comparison.png')

prompt = """Compare these two pie charts:
1. What are the main differences?
2. Which products increased/decreased?
3. What insights can you derive?"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, comp_image])
print(response.text)

## Visual Reasoning

In [ ]:
print("\n" + "="*80)
print("Visual Pattern Reasoning")
print("="*80)

# Create a visual pattern puzzle
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

patterns = [
    {'shape': 'circle', 'color': 'red', 'size': 0.3},
    {'shape': 'square', 'color': 'blue', 'size': 0.4},
    {'shape': 'circle', 'color': 'red', 'size': 0.5},
    None  # To be predicted
]

for i, (ax, pattern) in enumerate(zip(axes, patterns)):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    if pattern:
        if pattern['shape'] == 'circle':
            circle = plt.Circle((0.5, 0.5), pattern['size'], color=pattern['color'])
            ax.add_patch(circle)
        else:
            square = Rectangle(
                (0.5-pattern['size'], 0.5-pattern['size']),
                2*pattern['size'], 2*pattern['size'],
                color=pattern['color']
            )
            ax.add_patch(square)
    else:
        ax.text(0.5, 0.5, '?', fontsize=60, ha='center', va='center')
    
    ax.set_title(f'Position {i+1}')

plt.tight_layout()
plt.savefig('/tmp/pattern.png', dpi=150)


In [ ]:
print("\n" + "="*80)
print("Video Understanding")
print("="*80)

import time

# Upload video file (using available video)
video_path = '14801276_2160_3840_30fps.mp4'
print(f"Uploading video: {video_path}")

video_file = client.files.upload(path=video_path)
print(f"Video uploaded: {video_file.name}")

# Wait for processing
print("Processing video...")
while video_file.state == 'PROCESSING':
    time.sleep(2)
    video_file = client.files.get(name=video_file.name)
print(".", end="", flush=True)
print(f"\nVideo ready! State: {video_file.state}")

# Analyze video
prompt = 'Describe what happens in this video. What do you see?'
print(f"\nQuery: {prompt}\n")

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, video_file]
)
print(f"Response:\n{response.text}")

## Diagram Understanding

In [ ]:
print("\n" + "="*80)
print("Frame-by-Frame Video Analysis")
print("="*80)

# Reuse the uploaded video file from previous cell
# If running standalone, uncomment the upload code

queries = [
    "What are the main objects or subjects in this video?",
    "Describe any movements or actions you observe",
    "What is the setting or environment shown?"
]

for i, query in enumerate(queries, 1):
    print(f"\n{i}. Query: {query}")

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=[query, video_file]
    )
print(f"   Answer: {response.text}\n")

In [ ]:
plt.close()

flowchart_img = Image.open('/tmp/flowchart.png')

prompt = """Analyze this flowchart:
1. Describe the process flow
2. How many steps are there?
3. What type of process does this represent?
4. Are there any decision points?"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[prompt, flowchart_img])
print(response.text)

In [ ]:
print("\n" + "="*80)
print("Action Recognition in Videos")
print("="*80)

# Analyze specific actions in the video
action_queries = [
    "List all distinct visual elements or objects in chronological order as they appear",
    "Are there any movements or transitions in this video?",
    "Describe the overall composition and visual style"
]

for query in action_queries:
    print(f"\nQuery: {query}")

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=[query, video_file]
    )
print(f"Answer: {response.text}\n")
print("-" * 80)

In [ ]:
print("\n" + "="*80)
print("Action Recognition in Videos")
print("="*80)

video_file = client.files.upload(file='14801276_2160_3840_30fps.mp4')

# Wait for processing
while video_file.state == 'PROCESSING':
    time.sleep(2)
    video_file = client.files.get(name=video_file.name)

prompts = [
    # Simple action detection
    "Is someone walking, running, or standing still in this video?",

    # Multiple actions
    "List all distinct actions performed in chronological order",

    # Complex activities
    "Describe the cooking process shown in the video step by step",

    # Anomaly detection
    "Are there any unusual or unexpected actions in this video?",

    # Interaction analysis
    "Describe how the people in the video interact with each other"
]

for prompt in prompts:
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=[prompt, video_file]
    )
print(f"{prompt}\n{response.text}\n")


In [ ]:
from IPython.display import Audio

audio_path = "sample-audio.mp3"
Audio(audio_path)

In [ ]:
print("\n" + "="*80)
print("Audio Transcription")
print("="*80)

import time

# Upload audio file (using available WAV file)
audio_path = 'Test.wav'
print(f"Uploading audio: {audio_path}")

audio_file = client.files.upload(file=audio_path)
print(f"Audio uploaded: {audio_file.name}")

# Wait for processing
print("Processing audio...")
while audio_file.state == 'PROCESSING':
    time.sleep(1)
    audio_file = client.files.get(name=audio_file.name)
print(".", end="", flush=True)
print(f"\nAudio ready! State: {audio_file.state}")

# Transcribe
print("\nTranscription:\n")
response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[
        'Transcribe this audio exactly as spoken. Include any speech, sounds, or notable audio features.',
        audio_file
    ]
)
print(response.text)

## Frame-by-Frame Analysis

In [ ]:
print("\n" + "="*80)
print("Audio Content Analysis")
print("="*80)

# Analyze audio content beyond transcription
analysis_prompts = [
    "What type of audio is this? (speech, music, ambient, etc.)",
    "Describe the audio quality and any notable characteristics",
    "What is the overall tone or mood of this audio?"
]

for prompt in analysis_prompts:
    print(f"\nAnalysis: {prompt}")

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=[prompt, audio_file]
    )
print(f"Result: {response.text}\n")
print("-" * 80)

## Action Recognition

In [ ]:
print("\n" + "="*80)
print("PDF Document Understanding")
print("="*80)

import time

# Upload PDF (using available PDF file)
pdf_path = 'batra_nilmtk.pdf'
print(f"Uploading PDF: {pdf_path}")

pdf_file = client.files.upload(path=pdf_path)
print(f"PDF uploaded: {pdf_file.name}")

# Wait for processing
print("Processing PDF...")
while pdf_file.state == 'PROCESSING':
    time.sleep(2)
    pdf_file = client.files.get(name=pdf_file.name)
print(".", end="", flush=True)
print(f"\nPDF ready! State: {pdf_file.state}")

# Analyze document
prompts = [
    "What type of document is this? Provide a brief overview.",
    "What are the main sections or topics covered?",
    "Summarize the key points in 3-4 sentences."
]

for prompt in prompts:
    print(f"\nQuery: {prompt}")

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=[prompt, pdf_file]
    )
print(f"Answer: {response.text}\n")
print("-" * 80)

In [ ]:
print("\n" + "="*80)
print("Multi-Page PDF Data Extraction")
print("="*80)

pdf_file = client.files.upload(file='batra_nilmtk.pdf')

# Wait for processing
while pdf_file.state == 'PROCESSING':
    time.sleep(2)
    pdf_file = client.files.get(name=pdf_file.name)

extraction_prompt = '''Extract the following from this research paper:
{
  "title": "",
  "authors": [],
  "abstract": "",
  "keywords": [],
  "introduction": {
    "page": 0,
    "summary": ""
  },
  "methodology": {
    "page": 0,
    "summary": ""
  },
  "results": {
    "page": 0,
    "key_findings": []
  },
  "conclusions": "",
  "references_count": 0,
  "figures_count": 0,
  "tables_count": 0
}

Return valid JSON only.'''

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[extraction_prompt, pdf_file]
)



In [ ]:
print("\n" + "="*80)
print("Multi-Page PDF Data Extraction")
print("="*80)

# Extract structured information from the PDF
extraction_prompt = """Extract the following information from this PDF document:
1. Title/heading
2. Authors (if any)
3. Main topics or sections
4. Any key findings or conclusions
5. Number of pages (estimate)

Format your response as a structured summary."""

print("Extracting structured data from PDF...")
print()

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=[extraction_prompt, pdf_file]
)
print(response.text)
print("\n" + "="*80)
print("PDF analysis demonstrates:")
print("- Multi-page document understanding")
print("- Structure extraction (headings, sections)")
print("- Content summarization")
print("- Data extraction from mixed content (text, tables, figures)")

# Part 7: Advanced Features (10 tasks)

## Structured JSON Output

In [ ]:
print("\n" + "="*80)
print("Structured JSON Output")
print("="*80)

text = """Sarah Johnson, 34, is a Senior Data Scientist at TechCorp in San Francisco. 
She specializes in machine learning and has 8 years of experience. Her skills include 
Python, TensorFlow, and SQL. Contact: sarah.j@techcorp.com, (555) 987-6543."""

schema = {
    "name": " ",
    "age": 0,
    "title": "",
    "company": "",
    "location": "",
    "experience_years": 0,
    "skills": [],
    "contact": {
        "email": "",
        "phone": ""
    }
}

prompt = f"""Extract information and return valid JSON matching this schema:
{json.dumps(schema, indent=2)}

Text: {text}

Return ONLY the JSON object, no markdown formatting:"""

response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=prompt)
result = response.text.strip()

# Clean up markdown if present
if result.startswith('```'):
    result = '\n'.join(result.split('\n')[1:-1])
    if result.startswith('json'):
        result = result[4:]

try:
    parsed = json.loads(result.strip())
print("Extracted JSON:")
print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Raw output:\n{result}")
print(f"\nJSON parse error: {e}")

## Function Calling

In [ ]:
print("\n" + "="*80)
print("Function Calling / Tool Use")
print("="*80)

from google.genai import types

# Define the actual function implementation
def get_current_weather(location: str, unit: str = "celsius") -> dict:
    """Get the current weather for a location."""
    # Simulated weather data
    weather_data = {
        "london": {"temp": 15, "condition": "Cloudy", "humidity": 78},
        "new york": {"temp": 22, "condition": "Sunny", "humidity": 65},
        "tokyo": {"temp": 25, "condition": "Rainy", "humidity": 85},
        "paris": {"temp": 18, "condition": "Partly Cloudy", "humidity": 70},
    }

    data = weather_data.get(location.lower(), {"temp": 20, "condition": "Unknown", "humidity": 70})
    return {
        "location": location,
        "temperature": data["temp"],
        "unit": unit,
        "condition": data["condition"],
        "humidity": data["humidity"]
    }

# Define function schema for Gemini
get_weather_func = types.FunctionDeclaration(
    name="get_current_weather",
    description="Get the current weather in a location",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. London, New York, Tokyo"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit"
            }
        },
        "required": ["location"]
    }
)

# Create tool
weather_tool = types.Tool(function_declarations=[get_weather_func])

# Test query
query = "What's the weather like in London?"
print(f"Query: {query}\n")

# Call with tools
response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=query,
    config=types.GenerateContentConfig(
        tools=[weather_tool]
    )
)

# Check for function call
function_called = False
for part in response.candidates[0].content.parts:
    if hasattr(part, 'function_call'):
        function_called = True
        func_call = part.function_call
        print(f"Function called: {func_call.name}")
print(f"Arguments: {dict(func_call.args)}")

        # Execute the actual function
        result = get_current_weather(**dict(func_call.args))
print(f"Function result: {result}\n")

        # Return result to model for final response
        response2 = client.models.generate_content(
            model="gemini-2.0-flash-thinking-exp-1219",
            contents=[
                query,
                response.candidates[0].content,
                types.Content(
                    role="function",
                    parts=[types.Part.from_function_response(
                        name=func_call.name,
                        response={"result": result}
                    )]
                )
            ]
        )
print(f"Final response:\n{response2.text}")

if not function_called:
    print(f"Model responded directly without calling function:\n{response.text}")
print("\n" + "="*80)
print("Function calling allows the model to:")
print("- Request structured data from external sources")
print("- Perform calculations or operations")
print("- Access real-time information")
print("- Build agentic workflows")

## Code Execution

In [ ]:
print("\n" + "="*80)
print("Code Execution")
print("="*80)

from google.genai import types

# Enable code execution
code_execution_tool = types.Tool(code_execution={})

# Example 1: Fibonacci numbers
print("Example 1: Calculate Fibonacci numbers\n")
response = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents="Calculate the first 10 Fibonacci numbers and show them as a list",
    config=types.GenerateContentConfig(
        tools=[code_execution_tool]
    )
)
print("Response:")
print(response.text)
print()

# Example 2: Mathematical calculation
print("="*80)
print("Example 2: Solve quadratic equation\n")

response2 = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents="Solve the quadratic equation 2x² - 7x + 3 = 0. Show the solutions and verify them by substituting back.",
    config=types.GenerateContentConfig(
        tools=[code_execution_tool]
    )
)
print(response2.text)
print()

# Example 3: Statistical analysis
print("="*80)
print("Example 3: Statistical analysis\n")

response3 = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents="""Given this data: [12, 15, 18, 22, 25, 30, 35, 40, 45, 50]
Calculate and display:
- Mean
- Median
- Standard deviation
- Variance""",
    config=types.GenerateContentConfig(
        tools=[code_execution_tool]
    )
)
print(response3.text)
print()

# Example 4: Data visualization concept
print("="*80)
print("Example 4: Generate random data\n")

response4 = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents="Generate 20 random numbers between 1 and 100, then find the min, max, and average.",
    config=types.GenerateContentConfig(
        tools=[code_execution_tool]
    )
)
print(response4.text)
print("\n" + "="*80)
print("Code execution features:")
print(" Execute Python code safely in sandboxed environment")
print(" Perform calculations and verify results")
print(" Generate and analyze data")
print(" Solve mathematical problems")
print(" Model can write and run code to answer questions")
print("  Limited libraries available (basic Python only)")
print("  No file system or network access")

## Search Grounding

In [ ]:
print("\n" + "="*80)
print("Search Grounding (Google Search Integration)")
print("="*80)

from google.genai import types

# Enable Google Search grounding
google_search_tool = types.Tool(google_search={})

# Ask time-sensitive questions
queries = [
    "What are the latest developments in AI announced this month?",
    "What is the current stock price of NVIDIA?",
    "Who won the latest Nobel Prize in Physics?",
    "What are today's top technology news headlines?"
]

print("Using Google Search grounding for real-time information:\n")

for i, query in enumerate(queries, 1):
    print(f"{i}. Query: {query}")

    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash-thinking-exp-1219",
            contents=query,
            config=types.GenerateContentConfig(
                tools=[google_search_tool]
            )
        )
print(f"Answer: {response.text}")

        # Try to access grounding metadata if available
        if hasattr(response, 'grounding_metadata') and response.grounding_metadata:
            print(f" Response is grounded with search results")
            if hasattr(response.grounding_metadata, 'web_search_queries'):
                print(f"  Search queries used: {response.grounding_metadata.web_search_queries}")
print()

    except Exception as e:
        print(f"Note: {str(e)[:100]}")
print("Search grounding may not be available for all models/regions")
print()
        break

print("="*80)
print("Search Grounding Benefits:")
print(" Access to real-time, up-to-date information")
print(" Factually grounded responses with web sources")
print(" Citations and sources can be included")
print(" Overcomes knowledge cutoff limitations")
print(" Great for current events, stock prices, news, etc.")
print()
print("Note: Search grounding availability may vary by:")
print("- Model version")
print("- Geographic region")
print("- API tier/quota")

## Thinking Mode (Chain-of-Thought)

In [ ]:
print("\n" + "="*80)
print("Thinking Mode - Extended Reasoning")
print("="*80)

complex_problem = """A farmer has 17 sheep. All but 9 die. How many are left?
Explain your reasoning step by step."""

# Regular response
print("Without explicit thinking instructions:")
response_regular = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=complex_problem
)
print(response_regular.text)

# With chain-of-thought prompting
print("\n" + "="*80)
print("With chain-of-thought reasoning:")

cot_prompt = f"""Let's solve this step by step:

{complex_problem}

Think through each step carefully:
Step 1: Identify what we know
Step 2: Identify what the question is asking
Step 3: Analyze the wording carefully
Step 4: Calculate the answer
Step 5: Verify the answer makes sense"""

response_cot = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=cot_prompt
)
print(response_cot.text)

# More complex reasoning
print("\n" + "="*80)
print("Complex multi-step reasoning:")

logic_puzzle = """Three friends Alice, Bob, and Charlie have different jobs:
doctor, teacher, and engineer.
- The doctor is older than Alice
- Charlie is younger than the teacher
- Bob is not the youngest
- The engineer is the oldest

Who has which job? Explain your reasoning."""

response_puzzle = client.models.generate_content(
    model="gemini-2.0-flash-thinking-exp-1219",
    contents=f"Solve this logic puzzle step by step:\n\n{logic_puzzle}"
)
print(response_puzzle.text)

## Code Generation

In [ ]:
print("\n" + "="*80)
print("Code Generation")
print("="*80)

code_tasks = [
    {
        "language": "Python",
        "task": """Create a decorator that measures function execution time
        and logs it with the function name."""
    },
    {
        "language": "JavaScript",
        "task": """Create an async function that fetches data from multiple
        URLs in parallel and returns combined results."""
    },
    {
        "language": "SQL",
        "task": """Write a query to find the top 5 customers by total
        purchase amount in the last 30 days."""
    }
]

for task in code_tasks:
    prompt = f"""Write {task['language']} code for this task:

{task['task']}

Include:
- Clean, production-ready code
- Type hints/comments where appropriate
- Error handling
- A brief explanation
"""

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
print(f"\n{task['language']} Task: {task['task'][:50]}...")
print("-" * 80)
print(response.text)
print()

## Mathematical Reasoning

In [ ]:
print("\n" + "="*80)
print("Mathematical Problem Solving")
print("="*80)

math_problems = [
    {
        "type": "Calculus",
        "problem": "Find the derivative of f(x) = (3x² + 2x - 1) * e^x"
    },
    {
        "type": "Linear Algebra",
        "problem": """Find the eigenvalues of the matrix:
        [[2, 1],
         [1, 2]]"""
    },
    {
        "type": "Statistics",
        "problem": """Given data: [12, 15, 18, 22, 25, 30, 35]
        Calculate: mean, median, variance, and standard deviation"""
    },
    {
        "type": "Optimization",
        "problem": """A rectangular garden has perimeter of 60m.
        What dimensions maximize the area?"""
    }
]

for prob in math_problems:
    prompt = f"""Solve this {prob['type']} problem step by step:

{prob['problem']}

Show all work and explain each step."""

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
print(f"\n{prob['type']}: {prob['problem'][:50]}...")
print("-" * 80)
print(response.text)
print()

## Scientific Reasoning

## Creative Writing

In [ ]:
print("\n" + "="*80)
print("Creative Writing")
print("="*80)

creative_tasks = [
    "Write a haiku about artificial intelligence",
    "Write a limerick about programming",
    "Write a short story (3 paragraphs) about time travel",
    "Write a product description for an AI-powered coffee maker",
    "Write a motivational speech for a data science team"
]

for i, task in enumerate(creative_tasks, 1):
    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=task
    )
print(f"\n{i}. {task}")
print("-" * 80)
print(response.text)
print()

# Summary and Best Practices

##  What We Covered

This notebook demonstrated **35 tasks** across **all modalities** supported by Gemini 3 Pro:

### API Fundamentals
- Single vs batch processing
- Streaming responses
- Configuration and safety settings

### Text Modality (10 tasks)
 Sentiment analysis, classification, NER, summarization, QA, translation, completion, extraction, keywords, rewriting

### Vision Modality (8 tasks)
 Object counting, VQA, chart analysis, OCR, captioning, comparison, reasoning, diagrams

### Video Modality (3 tasks)
 Understanding, frame analysis, action recognition

### Audio Modality (2 tasks)
 Transcription, content analysis

### PDF Modality (2 tasks)
 Document analysis, multi-page extraction

### Advanced Features (10 tasks)
 Structured output, function calling, code execution, search grounding, thinking mode, long context, code generation, math, science, creative writing

##  Performance Tips

1. **For Best Results:**
   - Be specific in prompts
   - Provide examples when possible
   - Use appropriate temperature settings
   - Structure outputs with schemas

2. **For Efficiency:**
   - Use context caching for repeated queries
   - Batch similar requests
   - Stream long responses
   - Optimize token usage

3. **For Accuracy:**
   - Enable search grounding for current events
   - Use code execution for calculations
   - Request step-by-step reasoning
   - Validate structured outputs

##  Resources

- [Gemini API Documentation](https://ai.google.dev/gemini-api/docs)
- [Gemini 3 Pro Model Card](https://ai.google.dev/gemini-api/docs/models#gemini-3-pro)
- [Google AI Studio](https://aistudio.google.com/)
- [Pricing](https://ai.google.dev/pricing)
- [Safety Settings](https://ai.google.dev/gemini-api/docs/safety-settings)

##  Next Steps

1. Test with your own use cases
2. Explore production features (caching, batching)
3. Build multimodal applications
4. Experiment with thinking mode for complex reasoning
5. Leverage the 1M token context for large documents

Happy building! 